# Osio 6: Aineistojen yhdistäminen Pandasissa ja SQL Pythonissa

Tässä demossa harjoitellaan useissa aineistoissa olevan tiedon yhdistämistä ja analysointia.
  
Aluksi tietokannan taulut haetaan SQL:llä DataFrameihin ja tallennetaan CSV-tiedostoiksi, jotta Pandasia voidaan käyttää datan käsittelyyn.


## Pandasin `merge()`

Pandasissa DataFrameja yhdistetään `merge()`-toiminnolla yhteisen avainsarakkeen perusteella.

- `inner` säilyttää rivit, joille löytyy vastine molemmista aineistoista.
- `left` säilyttää kaikki vasemman DataFramen rivit.

Yhdistämisen jälkeen tarkistetaan aina aineiston koko, ensimmäiset rivit ja puuttuvat arvot ennen analyysin jatkamista.

## SQL Pythonissa

Pythonilla voidaan muodostaa yhteys suoraan esimerkiksi SQL Server-, Azure SQL-, PostgreSQL- tai MySQL-tietokantaan. SQL-kyselyn tulos voidaan lukea Pandasin DataFrameen ja käsittelyä voidaan jatkaa Pandasilla.

Käytännön syistä tässä demossa käytetään paikallista `Chinook_Sqlite.sqlite`-tiedostoa. Periaate on sama:

**Python → tietokantayhteys → SQL-kysely → Pandas DataFrame**

## Aineiston valmistelu: tietokannasta CSV-tiedostoiksi

Jotta voimme käyttää pelkästään Pandasia, niin haetaan aluksi tarvittavat taulut SQLite-tietokannasta ja tallennetaan ne CSV-tiedostoiksi. Tämä toimenpide vaaditaan myös palautettavassa harjoituksessa.

In [1]:
import pandas as pd
import sqlite3

In [2]:
yhteys = sqlite3.connect("~/shared/Chinook_Sqlite.sqlite")

Luetaan demossa tarvittavat taulut. `pd.read_sql_query()` suorittaa kyselyn ja palauttaa tuloksen DataFramena.

In [3]:
asiakkaat_kannasta = pd.read_sql_query("SELECT * FROM Customer", yhteys)
laskut_kannasta = pd.read_sql_query("SELECT * FROM Invoice", yhteys)
laskurivit_kannasta = pd.read_sql_query("SELECT * FROM InvoiceLine", yhteys)
kappaleet_kannasta = pd.read_sql_query("SELECT * FROM Track", yhteys)
musiikkilajit_kannasta = pd.read_sql_query("SELECT * FROM Genre", yhteys)

Tallennetaan DataFramet CSV-tiedostoiksi. `index=False` estää indeksin tallentumisen ylimääräiseksi sarakkeeksi.

In [4]:
asiakkaat_kannasta.to_csv("~/Customer.csv", index=False)
laskut_kannasta.to_csv("~/Invoice.csv", index=False)
laskurivit_kannasta.to_csv("~/InvoiceLine.csv", index=False)
kappaleet_kannasta.to_csv("~/Track.csv", index=False)
musiikkilajit_kannasta.to_csv("~/Genre.csv", index=False)

## Tehtävä 1a: Tutustutaan aineistoihin Pandasilla

Luetaan asiakkaat, laskut ja laskurivit CSV-tiedostoista.

In [5]:
asiakkaat = pd.read_csv("~/Customer.csv")
laskut = pd.read_csv("~/Invoice.csv")
laskurivit = pd.read_csv("~/InvoiceLine.csv")

In [6]:
display(asiakkaat.head())
display(laskut.head())
display(laskurivit.head())

,CustomerId,FirstName,LastName,Company,Address,City,State,Country,PostalCode,Phone,Fax,Email,SupportRepId
0,1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,+55 (12) 3923-5555,+55 (12) 3923-5566,luisg@embraer.com.br,3
1,2,Leonie,Köhler,NaN,Theodor-Heuss-Straße 34,Stuttgart,NaN,Germany,70174,+49 0711 2842222,NaN,leonekohler@surfeu.de,5
2,3,François,Tremblay,NaN,1498 rue Bélanger,Montréal,QC,Canada,H2G 1A7,+1 (514) 721-4711,NaN,ftremblay@gmail.com,3
3,4,Bjørn,Hansen,NaN,Ullevålsveien 14,Oslo,NaN,Norway,0171,+47 22 44 22 22,NaN,bjorn.hansen@yahoo.no,4
4,5,František,Wichterlová,JetBrains s.r.o.,Klanova 9/506,Prague,NaN,Czech Republic,14700,+420 2 4172 5555,+420 2 4172 5555,frantisekw@jetbrains.com,4


,InvoiceId,CustomerId,InvoiceDate,BillingAddress,BillingCity,BillingState,BillingCountry,BillingPostalCode,Total
0,1,2,2021-01-01 00:00:00,Theodor-Heuss-Straße 34,Stuttgart,NaN,Germany,70174,1.98
1,2,4,2021-01-02 00:00:00,Ullevålsveien 14,Oslo,NaN,Norway,0171,3.96
2,3,8,2021-01-03 00:00:00,Grétrystraat 63,Brussels,NaN,Belgium,1000,5.94
3,4,14,2021-01-06 00:00:00,8210 111 ST NW,Edmonton,AB,Canada,T6G 2C7,8.91
4,5,23,2021-01-11 00:00:00,69 Salem Street,Boston,MA,USA,2113,13.86


,InvoiceLineId,InvoiceId,TrackId,UnitPrice,Quantity
0,1,1,2,0.99,1
1,2,1,4,0.99,1
2,3,2,6,0.99,1
3,4,2,8,0.99,1
4,5,2,10,0.99,1


In [7]:
print("Asiakkaat:", asiakkaat.shape)
print("Laskut:", laskut.shape)
print("Laskurivit:", laskurivit.shape)
print("Yhteiset sarakkeet:")
print(asiakkaat.columns.intersection(laskut.columns).tolist())
print(laskut.columns.intersection(laskurivit.columns).tolist())

Asiakkaat: (59, 13)
Laskut: (412, 9)
Laskurivit: (2240, 5)
Yhteiset sarakkeet:
['CustomerId']
['InvoiceId']


## Tehtävä 1b: Tutustutaan tietoihin SQL:llä

Tutkitaan aluksi mitä tauluja kannassa on (vaikka se onkin kerrottu oletuksena jo aikaisemmin), taulujen rakenteet ja tämän jälkeen haetaan valitut asiakassarakkeet ja rajoitetaan tulos viiteen riviin.

In [8]:
kysely = """
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name
"""

pd.read_sql_query(kysely, yhteys)

,name
0,Album
1,Artist
2,Customer
3,Employee
4,Genre
5,Invoice
6,InvoiceLine
7,MediaType
8,Playlist
9,PlaylistTrack


In [9]:
kysely = """
PRAGMA table_info(Customer)
"""

pd.read_sql_query(kysely, yhteys)

#Tutki muiden taulujen rakennetta vaihtamalla taulun nimi.

,cid,name,type,notnull,dflt_value,pk
0,0,CustomerId,INTEGER,1,None,1
1,1,FirstName,NVARCHAR(40),1,None,0
2,2,LastName,NVARCHAR(20),1,None,0
3,3,Company,NVARCHAR(80),0,None,0
4,4,Address,NVARCHAR(70),0,None,0
5,5,City,NVARCHAR(40),0,None,0
6,6,State,NVARCHAR(40),0,None,0
7,7,Country,NVARCHAR(40),0,None,0
8,8,PostalCode,NVARCHAR(10),0,None,0
9,9,Phone,NVARCHAR(24),0,None,0


In [10]:
kysely = """
SELECT CustomerId, FirstName, LastName, Country
FROM Customer
LIMIT 5
"""
pd.read_sql_query(kysely, yhteys)

,CustomerId,FirstName,LastName,Country
0,1,Luís,Gonçalves,Brazil
1,2,Leonie,Köhler,Germany
2,3,François,Tremblay,Canada
3,4,Bjørn,Hansen,Norway
4,5,František,Wichterlová,Czech Republic


## Tehtävä 2a: Yhdistetään asiakkaat ja laskut Pandasilla

Yhdistetään aineistot `CustomerId`-sarakkeen perusteella. `validate="one_to_many"` tarkistaa oletuksen, että asiakas esiintyy asiakasaineistossa kerran mutta asiakkaalla voi olla monta laskua.

In [11]:
asiakkaat_laskut = pd.merge(
    asiakkaat,
    laskut,
    on="CustomerId",
    how="inner",
    validate="one_to_many"
)

Tarkistetaan yhdistämisen tulos ennen analyysiä. Rivimäärä kasvaa tarkoituksellisesti, koska asiakkaalla voi olla useita laskuja.

In [12]:
print("Asiakkaat:", asiakkaat.shape)
print("Laskut:", laskut.shape)
print("Yhdistetty:", asiakkaat_laskut.shape)
display(asiakkaat_laskut.head())
display(asiakkaat_laskut.isna().sum().sort_values(ascending=False).head(10))

Asiakkaat: (59, 13)
Laskut: (412, 9)
Yhdistetty: (412, 21)


,CustomerId,FirstName,LastName,Company,Address,City,State,Country,PostalCode,Phone,...,Email,SupportRepId,InvoiceId,InvoiceDate,BillingAddress,BillingCity,BillingState,BillingCountry,BillingPostalCode,Total
0,1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,+55 (12) 3923-5555,...,luisg@embraer.com.br,3,98,2022-03-11 00:00:00,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,3.98
1,1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,+55 (12) 3923-5555,...,luisg@embraer.com.br,3,121,2022-06-13 00:00:00,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,3.96
2,1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,+55 (12) 3923-5555,...,luisg@embraer.com.br,3,143,2022-09-15 00:00:00,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,5.94
3,1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,+55 (12) 3923-5555,...,luisg@embraer.com.br,3,195,2023-05-06 00:00:00,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,0.99
4,1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,+55 (12) 3923-5555,...,luisg@embraer.com.br,3,316,2024-10-27 00:00:00,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,1.98


Company              342
Fax                  328
State                202
BillingState         202
PostalCode            28
BillingPostalCode     28
Phone                  7
FirstName              0
CustomerId             0
Address                0
dtype: int64

In [13]:
asiakkaat_laskut[[
    "CustomerId", "FirstName", "LastName", "Country",
    "InvoiceId", "InvoiceDate", "Total"
]].head()

,CustomerId,FirstName,LastName,Country,InvoiceId,InvoiceDate,Total
0,1,Luís,Gonçalves,Brazil,98,2022-03-11 00:00:00,3.98
1,1,Luís,Gonçalves,Brazil,121,2022-06-13 00:00:00,3.96
2,1,Luís,Gonçalves,Brazil,143,2022-09-15 00:00:00,5.94
3,1,Luís,Gonçalves,Brazil,195,2023-05-06 00:00:00,0.99
4,1,Luís,Gonçalves,Brazil,316,2024-10-27 00:00:00,1.98


### Huomio: eri nimiset avainsarakkeet

Jos avainsarakkeilla on eri nimet, käytetään `left_on`- ja `right_on`-parametreja.

In [14]:
laskut_eri_avainnimella = laskut.rename(columns={"CustomerId": "AsiakasId"})

esimerkki_eri_avainnimista = pd.merge(
    asiakkaat,
    laskut_eri_avainnimella,
    left_on="CustomerId",
    right_on="AsiakasId",
    how="inner",
    validate="one_to_many"
)

esimerkki_eri_avainnimista[[
    "CustomerId", "AsiakasId", "FirstName", "LastName", "InvoiceId"
]].head()

,CustomerId,AsiakasId,FirstName,LastName,InvoiceId
0,1,1,Luís,Gonçalves,98
1,1,1,Luís,Gonçalves,121
2,1,1,Luís,Gonçalves,143
3,1,1,Luís,Gonçalves,195
4,1,1,Luís,Gonçalves,316


## Tehtävä 2b: Yhdistetään asiakkaat ja laskut SQL:llä

 Tehdään sama yhdistäminen `INNER JOIN` -liitoksella.

In [15]:
kysely = """
SELECT c.CustomerId, c.FirstName, c.LastName, c.Country,
       i.InvoiceId, i.InvoiceDate, i.Total
FROM Customer AS c
INNER JOIN Invoice AS i ON c.CustomerId = i.CustomerId
LIMIT 10
"""
pd.read_sql_query(kysely, yhteys)

,CustomerId,FirstName,LastName,Country,InvoiceId,InvoiceDate,Total
0,2,Leonie,Köhler,Germany,1,2021-01-01 00:00:00,1.98
1,4,Bjørn,Hansen,Norway,2,2021-01-02 00:00:00,3.96
2,8,Daan,Peeters,Belgium,3,2021-01-03 00:00:00,5.94
3,14,Mark,Philips,Canada,4,2021-01-06 00:00:00,8.91
4,23,John,Gordon,USA,5,2021-01-11 00:00:00,13.86
5,37,Fynn,Zimmermann,Germany,6,2021-01-19 00:00:00,0.99
6,38,Niklas,Schröder,Germany,7,2021-02-01 00:00:00,1.98
7,40,Dominique,Lefebvre,France,8,2021-02-01 00:00:00,1.98
8,42,Wyatt,Girard,France,9,2021-02-02 00:00:00,3.96
9,46,Hugh,O'Reilly,Ireland,10,2021-02-03 00:00:00,5.94


## Tehtävä 3a: Verrataan `inner`- ja `left`-liitoksia Pandasilla

In [16]:
inner_tulos = pd.merge(asiakkaat, laskut, on="CustomerId", how="inner")
left_tulos = pd.merge(asiakkaat, laskut, on="CustomerId", how="left")

print("Inner:", inner_tulos.shape)
print("Left:", left_tulos.shape)

asiakkaat_ilman_laskua = left_tulos[left_tulos["InvoiceId"].isna()]
asiakkaat_ilman_laskua[["CustomerId", "FirstName", "LastName", "InvoiceId"]]

Inner: (412, 21)
Left: (412, 21)


,CustomerId,FirstName,LastName,InvoiceId


## Tehtävä 3b: Verrataan SQL-liitoksia

 Etsitään `LEFT JOIN` -liitoksella asiakkaat, joilla ei ole laskua.

In [17]:
kysely = """
SELECT c.CustomerId, c.FirstName, c.LastName
FROM Customer AS c
LEFT JOIN Invoice AS i ON c.CustomerId = i.CustomerId
WHERE i.InvoiceId IS NULL
"""
pd.read_sql_query(kysely, yhteys)

,CustomerId,FirstName,LastName


## Tehtävä 4a: Yhdistetään kolme aineistoa Pandasilla

Lisätään yhdistettyyn asiakas- ja laskuaineistoon laskurivit `InvoiceId`-sarakkeen avulla.

In [18]:
asiakkaat_laskut_rivit = pd.merge(
    asiakkaat_laskut,
    laskurivit,
    on="InvoiceId",
    how="inner",
    validate="one_to_many"
)

In [19]:
print("Asiakkaat ja laskut:", asiakkaat_laskut.shape)
print("Laskurivit:", laskurivit.shape)
print("Kolmen aineiston yhdistelmä:", asiakkaat_laskut_rivit.shape)
display(asiakkaat_laskut_rivit.head())
display(asiakkaat_laskut_rivit.isna().sum().sort_values(ascending=False).head(10))

Asiakkaat ja laskut: (412, 21)
Laskurivit: (2240, 5)
Kolmen aineiston yhdistelmä: (2240, 25)


,CustomerId,FirstName,LastName,Company,Address,City,State,Country,PostalCode,Phone,...,BillingAddress,BillingCity,BillingState,BillingCountry,BillingPostalCode,Total,InvoiceLineId,TrackId,UnitPrice,Quantity
0,1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,+55 (12) 3923-5555,...,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,3.98,531,3247,1.99,1
1,1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,+55 (12) 3923-5555,...,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,3.98,532,3248,1.99,1
2,1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,+55 (12) 3923-5555,...,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,3.96,649,447,0.99,1
3,1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,+55 (12) 3923-5555,...,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,3.96,650,449,0.99,1
4,1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,+55 (12) 3923-5555,...,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,3.96,651,451,0.99,1


Company              1860
Fax                  1784
State                1100
BillingState         1100
PostalCode            152
BillingPostalCode     152
Phone                  38
CustomerId              0
FirstName               0
Country                 0
dtype: int64

Sama lasku esiintyy usealla rivillä, jos laskuun sisältyy useita kappaleita. Lasketaan laskurivin myynnin arvo.

In [20]:
asiakkaat_laskut_rivit["Rivin_myynti"] = (
    asiakkaat_laskut_rivit["UnitPrice"] * asiakkaat_laskut_rivit["Quantity"]
)
asiakkaat_laskut_rivit[["InvoiceId", "TrackId", "UnitPrice", "Quantity", "Rivin_myynti"]].head()

,InvoiceId,TrackId,UnitPrice,Quantity,Rivin_myynti
0,98,3247,1.99,1,1.99
1,98,3248,1.99,1,1.99
2,121,447,0.99,1,0.99
3,121,449,0.99,1,0.99
4,121,451,0.99,1,0.99


## Tehtävä 4b: Yhdistetään kolme taulua SQL:llä

In [21]:
kysely = """
SELECT c.CustomerId, c.FirstName, c.LastName,
       i.InvoiceId, i.InvoiceDate,
       il.TrackId, il.UnitPrice, il.Quantity,
       il.UnitPrice * il.Quantity AS Rivin_myynti
FROM Customer AS c
INNER JOIN Invoice AS i ON c.CustomerId = i.CustomerId
INNER JOIN InvoiceLine AS il ON i.InvoiceId = il.InvoiceId
LIMIT 10
"""
pd.read_sql_query(kysely, yhteys)

,CustomerId,FirstName,LastName,InvoiceId,InvoiceDate,TrackId,UnitPrice,Quantity,Rivin_myynti
0,2,Leonie,Köhler,1,2021-01-01 00:00:00,2,0.99,1,0.99
1,2,Leonie,Köhler,1,2021-01-01 00:00:00,4,0.99,1,0.99
2,4,Bjørn,Hansen,2,2021-01-02 00:00:00,6,0.99,1,0.99
3,4,Bjørn,Hansen,2,2021-01-02 00:00:00,8,0.99,1,0.99
4,4,Bjørn,Hansen,2,2021-01-02 00:00:00,10,0.99,1,0.99
5,4,Bjørn,Hansen,2,2021-01-02 00:00:00,12,0.99,1,0.99
6,8,Daan,Peeters,3,2021-01-03 00:00:00,16,0.99,1,0.99
7,8,Daan,Peeters,3,2021-01-03 00:00:00,20,0.99,1,0.99
8,8,Daan,Peeters,3,2021-01-03 00:00:00,24,0.99,1,0.99
9,8,Daan,Peeters,3,2021-01-03 00:00:00,28,0.99,1,0.99


## Tehtävä 5a: Analysoidaan asiakkaiden ostoksia Pandasilla

In [22]:
asiakkaiden_ostokset = (
    asiakkaat_laskut
    .groupby(["CustomerId", "FirstName", "LastName"], as_index=False)
    .agg(Ostokset_yhteensa=("Total", "sum"),
         Laskujen_maara=("InvoiceId", "nunique"))
    .sort_values("Ostokset_yhteensa", ascending=False)
)
asiakkaiden_ostokset["Ostokset_yhteensa"] = asiakkaiden_ostokset["Ostokset_yhteensa"].round(2)
asiakkaiden_ostokset.head(10)

,CustomerId,FirstName,LastName,Ostokset_yhteensa,Laskujen_maara
5,6,Helena,Holý,49.62,7
25,26,Richard,Cunningham,47.62,7
56,57,Luis,Rojas,46.62,7
44,45,Ladislav,Kovács,45.62,7
45,46,Hugh,O'Reilly,45.62,7
27,28,Julia,Barnett,43.62,7
36,37,Fynn,Zimmermann,43.62,7
23,24,Frank,Ralston,43.62,7
6,7,Astrid,Gruber,42.62,7
24,25,Victor,Stevens,42.62,7


## Tehtävä 5b: Sama analyysi SQL:llä

In [23]:
kysely = """
SELECT c.CustomerId, c.FirstName, c.LastName,
       ROUND(SUM(i.Total), 2) AS Ostokset_yhteensa,
       COUNT(DISTINCT i.InvoiceId) AS Laskujen_maara
FROM Customer AS c
INNER JOIN Invoice AS i ON c.CustomerId = i.CustomerId
GROUP BY c.CustomerId, c.FirstName, c.LastName
ORDER BY Ostokset_yhteensa DESC
LIMIT 10
"""
pd.read_sql_query(kysely, yhteys)

,CustomerId,FirstName,LastName,Ostokset_yhteensa,Laskujen_maara
0,6,Helena,Holý,49.62,7
1,26,Richard,Cunningham,47.62,7
2,57,Luis,Rojas,46.62,7
3,45,Ladislav,Kovács,45.62,7
4,46,Hugh,O'Reilly,45.62,7
5,24,Frank,Ralston,43.62,7
6,28,Julia,Barnett,43.62,7
7,37,Fynn,Zimmermann,43.62,7
8,7,Astrid,Gruber,42.62,7
9,25,Victor,Stevens,42.62,7


## Tehtävä 6a: Selvitetään myynti maittain Pandasilla

In [24]:
myynti_maittain = (
    asiakkaat_laskut
    .groupby("Country", as_index=False)
    .agg(Myynti_yhteensa=("Total", "sum"),
         Laskujen_maara=("InvoiceId", "nunique"),
         Asiakkaiden_maara=("CustomerId", "nunique"))
    .sort_values("Myynti_yhteensa", ascending=False)
)
myynti_maittain["Myynti_yhteensa"] = myynti_maittain["Myynti_yhteensa"].round(2)
myynti_maittain.head(10)

,Country,Myynti_yhteensa,Laskujen_maara,Asiakkaiden_maara
22,USA,523.06,91,13
5,Canada,303.96,56,8
10,France,195.10,35,5
4,Brazil,190.10,35,5
11,Germany,156.48,28,4
23,United Kingdom,112.86,21,3
7,Czech Republic,90.24,14,2
19,Portugal,77.24,14,2
13,India,75.26,13,2
6,Chile,46.62,7,1


## Tehtävä 6b: Sama analyysi SQL:llä

In [25]:
kysely = """
SELECT c.Country,
       ROUND(SUM(i.Total), 2) AS Myynti_yhteensa,
       COUNT(DISTINCT i.InvoiceId) AS Laskujen_maara,
       COUNT(DISTINCT c.CustomerId) AS Asiakkaiden_maara
FROM Customer AS c
INNER JOIN Invoice AS i ON c.CustomerId = i.CustomerId
GROUP BY c.Country
ORDER BY Myynti_yhteensa DESC
"""
pd.read_sql_query(kysely, yhteys).head(10)

,Country,Myynti_yhteensa,Laskujen_maara,Asiakkaiden_maara
0,USA,523.06,91,13
1,Canada,303.96,56,8
2,France,195.10,35,5
3,Brazil,190.10,35,5
4,Germany,156.48,28,4
5,United Kingdom,112.86,21,3
6,Czech Republic,90.24,14,2
7,Portugal,77.24,14,2
8,India,75.26,13,2
9,Chile,46.62,7,1


## Tehtävä 7a: Selvitetään eniten myyntiä tuottaneet musiikkilajit Pandasilla

Luetaan kappaleet ja musiikkilajit CSV-tiedostoista.

In [26]:
kappaleet = pd.read_csv("Track.csv")
musiikkilajit = pd.read_csv("Genre.csv")

Valitaan tarvittavat sarakkeet ja nimetään samannimiset `Name`-sarakkeet selkeästi ennen yhdistämistä.

In [27]:
laskurivit_valitut = laskurivit[["InvoiceLineId", "InvoiceId", "TrackId", "UnitPrice", "Quantity"]]
kappaleet_valitut = kappaleet[["TrackId", "Name", "GenreId"]].rename(columns={"Name": "Kappale"})
musiikkilajit_valitut = musiikkilajit[["GenreId", "Name"]].rename(columns={"Name": "Musiikkilaji"})

In [28]:
myydyt_kappaleet = pd.merge(
    laskurivit_valitut, kappaleet_valitut,
    on="TrackId", how="inner", validate="many_to_one"
)
myynti_ja_lajit = pd.merge(
    myydyt_kappaleet, musiikkilajit_valitut,
    on="GenreId", how="inner", validate="many_to_one"
)
myynti_ja_lajit["Rivin_myynti"] = myynti_ja_lajit["UnitPrice"] * myynti_ja_lajit["Quantity"]

myynti_lajeittain = (
    myynti_ja_lajit.groupby("Musiikkilaji", as_index=False)
    .agg(Myynti_yhteensa=("Rivin_myynti", "sum"), Myyty_maara=("Quantity", "sum"))
    .sort_values("Myynti_yhteensa", ascending=False)
)
myynti_lajeittain["Myynti_yhteensa"] = myynti_lajeittain["Myynti_yhteensa"].round(2)
myynti_lajeittain.head(10)

,Musiikkilaji,Myynti_yhteensa,Myyty_maara
17,Rock,826.65,835
12,Latin,382.14,386
13,Metal,261.36,264
1,Alternative & Punk,241.56,244
22,TV Shows,93.53,47
11,Jazz,79.20,80
2,Blues,60.39,61
6,Drama,57.71,29
4,Classical,40.59,41
15,R&B/Soul,40.59,41


## Tehtävä 7b: Sama analyysi SQL:llä

In [29]:
kysely = """
SELECT g.Name AS Musiikkilaji,
       ROUND(SUM(il.UnitPrice * il.Quantity), 2) AS Myynti_yhteensa,
       SUM(il.Quantity) AS Myyty_maara
FROM InvoiceLine AS il
INNER JOIN Track AS t ON il.TrackId = t.TrackId
INNER JOIN Genre AS g ON t.GenreId = g.GenreId
GROUP BY g.GenreId, g.Name
ORDER BY Myynti_yhteensa DESC
"""
pd.read_sql_query(kysely, yhteys).head(10)

,Musiikkilaji,Myynti_yhteensa,Myyty_maara
0,Rock,826.65,835
1,Latin,382.14,386
2,Metal,261.36,264
3,Alternative & Punk,241.56,244
4,TV Shows,93.53,47
5,Jazz,79.20,80
6,Blues,60.39,61
7,Drama,57.71,29
8,R&B/Soul,40.59,41
9,Classical,40.59,41


## Lopuksi

Suljetaan tietokantayhteys.

In [30]:
yhteys.close()